# 決算発表月のリターンで見る、日本の収益性ファクター（TOPIXユニバース）

**仮説**：日本では高収益が長続きしないのに、市場はそれが続くと見込んで割高に評価している。
そのため、**決算発表で収益性の低下が明らかになったときに株価が下がり**、収益性ファクターの効果が打ち消される。

**なぜ決算発表月を見るのか**：割引率（要求リターン）の仮定に頼らずにミスプライシングを検証するため。
リスクが原因なら、リターンの差が決算発表の時期に集中する理由はない。
集中していれば、決算で「期待が裏切られた」ことが原因だと解釈できる
（La Porta, Lakonishok, Shleifer & Vishny 1997；Engelberg, McLean & Pontiff 2018 と同じ考え方）。

| 章 | 内容 |
|---|---|
| 0〜3 | 設定・データ取得・ユニバース作成（TOPIX × 財務 × リターン × 決算発表日の結合） |
| 4 | 財務指標の作成（ROE・資産成長・アクルーアル等）と時点整合 |
| 5 | 前提の確認：高ROE企業のうち、資産成長が高い企業ほど高収益が続かないか |
| 6 | 決算発表月とそれ以外の月のリターン比較（収益性階層別） |
| 7 | 収益性ロングショートの決算発表月への集中 |
| 8 | 月次クロスセクション回帰（収益性 × 決算発表月） |
| 9 | 高ROE企業を「続きそう / 続かなそう」に分けた比較（9-2: リターン調整方法の比較） |
| 10 | 決算で明らかになった ROE の変化とリターンの関係 |
| 11 | 期間別 |
| 12 | まとめ（自動抽出） |

**制約**：リターンは月次なので、決算発表「日」ではなく決算発表「月」で見る（日次リターンがあれば精度が上がる）。

## 0. 設定

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
# ===== パラメータ =====
START_YM = 200001               # 分析開始（組成月, YYYYMM）
END_YM = 999912                 # 分析終了

TIER_CUTS = (0.3, 0.7)          # ROEの国内パーセンタイル：〜30%=低収益 / 30〜70%=中収益 / 70%〜=高収益
CTYPE_PRIORITY = [3, 1, 2]      # 会計基準の優先順位（3: IFRS, 1: SEC, 2: 日本式）
MAX_STALE_MONTHS = 15           # 最新の本決算から何か月までを有効とするか
EXCLUDE_INDUSTRY_KEYWORDS = ["銀行", "証券", "保険", "その他金融"]   # 東証33業種名に含まれていれば除外

MIN_STOCKS = 5                  # グループ平均を計算する最小銘柄数
NW_LAGS = 6                     # Newey-West のラグ
PERIODS = {"2000-2014": (2000, 2014), "2015-2022": (2015, 2022), "2023-": (2023, 2100)}

# 主な分析で使うリターンの調整方法（日米で作り方をそろえるため、既定は B/P を使わない市場調整）
#   "mkt": 同月の等ウェイト平均を引く / "size": サイズ3分位の平均を引く / "size_bp": サイズ×B/P の3×3セル平均を引く
RET_ADJ = "mkt"

USE_DUMMY_DATA = False          # True で架空データによる動作確認（結果に意味はない）

## 1. データ取得関数

`fetch_data` と `get_data` は社内DB接続用の関数です。import 文を記入してください。
（どちらか一方しかない場合は、同じ関数を両方の名前で使います）

In [ ]:
# 社内DB接続関数の読み込み（環境に合わせて書き換えてください）
# from xxxxx import fetch_data, get_data

if not USE_DUMMY_DATA:
    if "fetch_data" not in globals() and "get_data" not in globals():
        raise NameError("fetch_data / get_data が定義されていません。上の import 文を記入してください。")
    if "get_data" not in globals():
        get_data = fetch_data
    if "fetch_data" not in globals():
        fetch_data = get_data

In [ ]:
# PL項目の取得
def get_pl_data():
    PL_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]        -- 決算期 (int yyyymm)
        ,[CTYPE]        -- 会計基準 (1: SEC 2: 日本式 3: IFRS)
        ,[GPLS]         -- 売上総利益
        ,[OPINCM]       -- 営業利益
        ,[CHAGE]        -- 支払利息
        ,[PLCURREN]     -- 為替換算調整勘定，のちのちの分析で必要かもなので一応
        ,[NETINCM]      -- 当期純利益
        ,[RD]           -- 研究開発費
    FROM [IDSQE].[dbo].[NRIFIN_CON_PL4]
    """
    df = fetch_data(PL_QUERY, 'idsqe')
    # 欠損値は-999999998
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# BS項目の取得
def get_bs_data():
    BS_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]        -- 決算期 (int yyyymm)
        ,[CTYPE]        -- 会計基準 (1: SEC 2: 日本式 3: IFRS)
        ,[CASH]         -- 現金・預金，のちのちの分析で必要かもなので一応
        ,[ACRC]         -- 売上債権，アクルーアルの計算要
        ,[INVENT]       -- 棚卸資産
        ,[INVSCRT]      -- 投資有価証券，別の分析でのちほど用いる
        ,[TOTALBS]      -- 資産合計
        ,[ACPY]         -- 買入債務
        ,[CAPSTTL]      -- 株主資本合計
        ,[NTASTTL]      -- 純資産合計，株主資本合計が欠損しているときに用いる
        ,[EQTY]         -- 資本合計，株主資本合計が欠損しているとき
    FROM [IDSQE].[dbo].[NRIFIN_CON_BS4]
    """
    df = fetch_data(BS_QUERY, 'idsqe')
    # 欠損値は-999999998
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 決算詳細
def get_accounting_details():
    DETAILS_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]            -- 決算期 int yyyymm
        ,[CTYPE]            -- 会計基準 1: SEC式 2: 日本式 3: IFRS式
        ,[SETTLEMENT_FLG]   -- 本中区分 1: 本決算 2: 中間決算 3: 第1四半期決算 4: 第3四半期決算
        ,[FDATE]            -- 決算年月日 char yyyymmdd
        ,[NMONTH]           -- 決算月数 通常12
        ,[ANNOUNCE]         -- 決算発表日，[RCVD]より欠損(-999999998)が多い
        ,[RCVD]             -- 決算短信入手日
    FROM [IDSQE].[dbo].[NRIFIN_CON_AT4]
    """
    df = fetch_data(DETAILS_QUERY, 'idsqe')
    # 欠損値は-999999998
    # 決算発表、短信入手日どちらも欠損している場合は、決算期プラス2か月する（日本の法律）
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 会社属性
# NRI_CODE -> 証券コード
def get_nri_to_ticker_table():
    TABLE_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]        -- int yyyymmdd
        ,[TO_DATE]          -- int yyyymmdd
        ,[ITEM_DATA]        -- 証券コード
    FROM [IDSQE].[dbo].[COMPANY_CODE]
    WHERE ITEM = 'OLDCD'            -- 証券コード
    """
    df = fetch_data(TABLE_QUERY, 'idsqe')
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 業種コード
def get_industry():
    IND_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]        -- int yyyymmdd
        ,[TO_DATE]          -- int yyyymmdd
        ,c.[ITEM_DATA]
        ,n.[IND_NAME]
    FROM [IDSQE].[dbo].[COMPANY_INDUSTRY_CLASS] c
    JOIN [IDSQE].[dbo].[INDUSTRY_CLASS_NAME] n ON c.ITEM = n.IND
    WHERE [ITEM] = 'IND33'   -- 東証33業種
    """
    df = fetch_data(IND_QUERY, 'idsqe')
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# TOPIX構成銘柄
def get_topix_data():
    query = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]
        ,[TO_DATE]
        ,[ITEM]
        ,[ITEM_DATA]
    FROM [IDSQE].[dbo].[STOCK_ISSUE_INDEX]
    WHERE [ITEM] = 'TOPIX';
    """
    return get_data(query, "idsqe")


# 時価総額の取得
def get_market_cap_data():
    # 月次時価総額と、各月末時点で有効な証券コードを取得
    query = """
    SELECT
        prc.[DATE] AS [date],
        prc.[CAP] AS market_cap,
        SUBSTRING(RTRIM(id.[ID]), 3, LEN(RTRIM(id.[ID])))
            AS security_code
    FROM [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_M_PRC] AS prc
    CROSS APPLY (
        -- YYYYMMから月末日（YYYYMMDD）を作成
        SELECT CONVERT(
            int,
            CONVERT(
                char(8),
                EOMONTH(DATEFROMPARTS(
                    prc.[DATE] / 100,
                    prc.[DATE] % 100,
                    1
                )),
                112
            )
        ) AS month_end
    ) AS d
    INNER JOIN [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_ID] AS id
        ON prc.[BID] = id.[BID]
        AND id.[IDTYPE] = 'LOCALID'
        AND id.[FDATE] <= d.[month_end]
        AND (id.[TDATE] >= d.[month_end] OR id.[TDATE] IS NULL);
    """
    df = get_data(query, "riskmodels")

    # 証券コードと数値列を整形
    df["security_code"] = df["security_code"].astype("string").str.strip().str.upper()
    df["date"] = pd.to_numeric(df["date"], errors="raise").astype(int)
    df["market_cap"] = (
        pd.to_numeric(df["market_cap"], errors="raise")
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
    )
    # 月次Period列を追加
    df["month"] = pd.to_datetime(df["date"].astype(str), format="%Y%m").dt.to_period("M")
    return df


# 銘柄リターンの取得
def get_return_data():
    # 月次リターンと、各月初時点で有効な証券コードを取得
    query = """
    SELECT
        rtn.[DATE],
        SUBSTRING(RTRIM(id.[ID]), 3, LEN(RTRIM(id.[ID])))
            AS security_code,
        rtn.[RTN]
    FROM [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_M_RTN] AS rtn
    INNER JOIN [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_ID] AS id
        ON rtn.[BID] = id.[BID]
    WHERE id.[IDTYPE] = 'LOCALID'
      AND TRY_CONVERT(date, CONCAT(rtn.[DATE], '01'), 112)
          >= TRY_CONVERT(date, CONVERT(varchar(8), id.[FDATE]), 112)
      AND (
          id.[TDATE] IS NULL
          OR TRY_CONVERT(date, CONCAT(rtn.[DATE], '01'), 112)
             <= TRY_CONVERT(date, CONVERT(varchar(8), id.[TDATE]), 112)
      );
    """
    df = get_data(query, "riskmodels")

    # 証券コード・年月を整形し、リターンを%から小数へ変換
    df["security_code"] = df["security_code"].astype("string").str.strip().str.upper()
    dates = pd.to_numeric(df["DATE"], errors="raise").astype(int)
    df["month"] = pd.to_datetime(dates.astype(str), format="%Y%m").dt.to_period("M")
    df["ret"] = pd.to_numeric(df["RTN"], errors="raise").astype(float) / 100
    return df[["security_code", "month", "ret"]].copy()

In [ ]:
def make_dummy_jp(n=600, y0=2000, y1=2024, seed=0):
    """動作確認用の架空データ（USE_DUMMY_DATA=True のときのみ使用）。各取得関数の戻り値と同じ形。分析上の意味はない。"""
    rng = np.random.default_rng(seed)
    nri = [f"N{i:05d}" for i in range(n)]
    codes = [str(1300 + i) for i in range(n)]
    inds = ["水産・農林業", "建設業", "食料品", "化学", "医薬品", "鉄鋼", "機械", "電気機器", "輸送用機器",
            "卸売業", "小売業", "銀行業", "証券、商品先物取引業", "保険業", "情報・通信業", "サービス業"]
    exit_dates = np.where(rng.random(n) < 0.15, rng.integers(2005, 2024, n) * 10000 + 630, 99991231)
    topix = pd.DataFrame({"NRI_CODE": nri, "FROM_DATE": 20000101, "TO_DATE": exit_dates, "ITEM": "TOPIX", "ITEM_DATA": "1"})
    code_tbl = pd.DataFrame({"NRI_CODE": nri, "FROM_DATE": 19000101, "TO_DATE": 99991231, "ITEM_DATA": codes})
    ind_pick = rng.choice(len(inds), n)
    ind = pd.DataFrame({"NRI_CODE": nri, "FROM_DATE": 19000101, "TO_DATE": 99991231,
                        "ITEM_DATA": [f"{k:04d}" for k in ind_pick], "IND_NAME": [inds[k] for k in ind_pick]})
    pl_rows, bs_rows, det_rows = [], [], []
    base_roe = rng.normal(0.07, 0.04, n)
    roe = base_roe.copy()
    ta = rng.lognormal(np.log(3e5), 1.0, n)
    eq = ta * rng.uniform(0.3, 0.6, n)
    for y in range(y0 - 1, y1 + 1):
        ag = rng.normal(0.04, 0.08, n) + 0.8 * (roe - 0.07)
        roe = 0.6 * roe + 0.4 * base_roe - 0.3 * (ag - 0.04) + rng.normal(0, 0.03, n)
        ta = ta * (1 + ag)
        ni = roe * eq
        eq = eq + 0.6 * ni
        ft = y * 100 + 3
        for i in range(n):
            pl_rows.append((nri[i], ft, 2, ni[i] * 4, ni[i] * 1.6, ni[i] * 0.05, -999999998, ni[i], ta[i] * 0.01))
            bs_rows.append((nri[i], ft, 2, ta[i] * 0.1, ta[i] * 0.15, ta[i] * 0.1, ta[i] * 0.05, ta[i],
                            ta[i] * 0.1, eq[i], eq[i] * 1.02, -999999998))
            day = int(rng.integers(8, 28))
            det_rows.append((nri[i], ft, 2, 1, f"{y}0331", 12, (y * 100 + 5) * 100 + day, (y * 100 + 5) * 100 + day))
            for flg, mm, yy in ((3, 8, y), (2, 11, y), (4, 2, y + 1)):
                d_ = (yy * 100 + mm) * 100 + int(rng.integers(5, 15))
                det_rows.append((nri[i], ft + 100 if flg != 1 else ft, 2, flg, f"{y}0331", 12,
                                 d_ if rng.random() > 0.3 else -999999998, d_))
    pl = pd.DataFrame(pl_rows, columns=["NRI_CODE", "FTERM", "CTYPE", "GPLS", "OPINCM", "CHAGE", "PLCURREN", "NETINCM", "RD"])
    bs = pd.DataFrame(bs_rows, columns=["NRI_CODE", "FTERM", "CTYPE", "CASH", "ACRC", "INVENT", "INVSCRT", "TOTALBS",
                                        "ACPY", "CAPSTTL", "NTASTTL", "EQTY"])
    det = pd.DataFrame(det_rows, columns=["NRI_CODE", "FTERM", "CTYPE", "SETTLEMENT_FLG", "FDATE", "NMONTH", "ANNOUNCE", "RCVD"])
    months = pd.period_range(f"{y0}-01", f"{y1}-12", freq="M")
    cap = rng.lognormal(np.log(1e5), 1.2, n)
    mrows, rrows = [], []
    for m in months:
        r = rng.normal(0.006, 0.05) + rng.normal(0, 0.08, n)
        cap = cap * (1 + r)
        mrows.append(pd.DataFrame({"date": int(m.strftime("%Y%m")), "market_cap": cap, "security_code": codes, "month": m}))
        rrows.append(pd.DataFrame({"security_code": codes, "month": m, "ret": r}))
    mcap = pd.concat(mrows, ignore_index=True)
    mcap["security_code"] = mcap["security_code"].astype("string")
    ret = pd.concat(rrows, ignore_index=True)
    ret["security_code"] = ret["security_code"].astype("string")
    ret = ret[rng.random(len(ret)) > 0.003]          # 上場廃止などによる欠損を模擬
    return dict(pl=pl, bs=bs, details=det, code=code_tbl, ind=ind, topix=topix, mcap=mcap, ret=ret)

In [ ]:
if USE_DUMMY_DATA:
    raw = make_dummy_jp()
else:
    raw = dict(
        pl=get_pl_data(), bs=get_bs_data(), details=get_accounting_details(),
        code=get_nri_to_ticker_table(), ind=get_industry(), topix=get_topix_data(),
        mcap=get_market_cap_data(), ret=get_return_data(),
    )
for k, v in raw.items():
    print(f"{k:8s}: {len(v):>10,} 行  列={list(v.columns)}")

## 2. 共通の道具

- 月は整数の通し番号 `mi`（1970年1月=0）で扱い、結合を速く確実にする
- 証券コードは前後空白除去・大文字化し、5桁で末尾が0のもの（チェックデジット付き）は先頭4桁にそろえる
  （結合率を3章で表示するので、低い場合はここを調整）
- 期間付きテーブル（TOPIX構成・証券コード・業種）は「**月末時点で有効か**」で月次に展開する

In [ ]:
MISSING = -999999990


def num(s):
    x = pd.to_numeric(s, errors="coerce")
    return x.mask(x <= MISSING)


def ym_to_mi(ym):
    ym = pd.to_numeric(ym, errors="coerce")
    return (ym // 100 - 1970) * 12 + ym % 100 - 1


def ymd_to_mi(ymd):
    return ym_to_mi(pd.to_numeric(ymd, errors="coerce") // 100)


def period_to_mi(p):
    return (p.dt.year - 1970) * 12 + p.dt.month - 1


def mi_to_ts(mi):
    mi = pd.Series(mi).astype(int)
    return pd.to_datetime(pd.DataFrame({"year": mi // 12 + 1970, "month": mi % 12 + 1, "day": 1}))


def mi_to_ym(mi):
    mi = np.asarray(mi).astype(int)
    return (mi // 12 + 1970) * 100 + mi % 12 + 1


def ymd_to_date(ymd):
    x = num(ymd)
    x = x.where((x > 19000101) & (x < 21001231))
    return pd.to_datetime(x.astype("Int64").astype("string"), format="%Y%m%d", errors="coerce")


def norm_code(s):
    s = s.astype("string").str.strip().str.upper()
    five = (s.str.len() == 5) & s.str.endswith("0")
    return s.where(~five, s.str[:4])


def expand_intervals(df, keys, from_col="FROM_DATE", to_col="TO_DATE", lo=None, hi=None):
    """期間 [FROM, TO] を「月末時点で有効な月」の行に展開する"""
    f = num(df[from_col]).fillna(19000101)
    t = num(df[to_col])
    t = t.where((t > 0) & (t < 21001231), 21001231)
    t_date = ymd_to_date(t)
    start = ymd_to_mi(f)                                            # 開始月の月末には有効
    end = ymd_to_mi(t) - (~t_date.dt.is_month_end.fillna(True)).astype(int)   # 月末より前に終われば前月まで
    if lo is not None:
        start = start.clip(lower=lo)
    if hi is not None:
        end = end.clip(upper=hi)
    n = (end - start + 1).clip(lower=0).fillna(0).astype(int).to_numpy()
    idx = np.repeat(np.arange(len(df)), n)
    offs = np.arange(n.sum()) - np.repeat(np.cumsum(n) - n, n)
    out = df.iloc[idx][keys].reset_index(drop=True)
    out["mi"] = np.repeat(start.to_numpy(), n).astype(int) + offs
    out["_from"] = np.repeat(f.to_numpy(), n)
    return out


def nw_tstat(x, lags=NW_LAGS):
    x = pd.Series(x).dropna().to_numpy(float)
    n = len(x)
    if n < 3:
        return np.nan
    e = x - x.mean()
    s = e @ e / n
    for l in range(1, min(lags, n - 1) + 1):
        s += 2 * (1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n
    return x.mean() / np.sqrt(s / n) if s > 0 else np.nan


def ts_stats(s, scale=100):
    s = pd.Series(s).dropna()
    return pd.Series({"平均(%/月)": s.mean() * scale, "t値(NW)": nw_tstat(s), "月数": len(s)})

## 3. ユニバースの作成（TOPIX × 証券コード × 時価総額 × リターン × 業種）

- 組成月 t の月末に TOPIX に含まれる企業（金融業を除く）
- 時価総額は月 t、**リターンは翌月 t+1**（月 t の証券コードで取得するので、TOPIX から外れた後のリターンも使う）
- 翌月リターンが無い銘柄（上場廃止など）は、ポートフォリオでは残りの銘柄に組成時ウェイトで再配分、回帰では除外

In [ ]:
_data_last = int(period_to_mi(raw["ret"]["month"]).max())
START_MI, END_MI = int(ym_to_mi(START_YM)), min(int(ym_to_mi(min(END_YM, 210012))), _data_last)

# TOPIX構成（月末時点）
topix_m = expand_intervals(raw["topix"].assign(NRI_CODE=lambda x: x["NRI_CODE"].astype(str).str.strip()),
                           ["NRI_CODE"], lo=START_MI, hi=END_MI).drop(columns="_from").drop_duplicates()

# 証券コード（月末時点で有効なもの。複数あれば開始日が新しい方）
code_m = expand_intervals(raw["code"].assign(NRI_CODE=lambda x: x["NRI_CODE"].astype(str).str.strip()),
                          ["NRI_CODE", "ITEM_DATA"], lo=START_MI, hi=END_MI)
code_m = (code_m.sort_values("_from").drop_duplicates(["NRI_CODE", "mi"], keep="last")
          .rename(columns={"ITEM_DATA": "security_code"}).drop(columns="_from"))
code_m["security_code"] = norm_code(code_m["security_code"])

# 業種（東証33業種）
ind_m = expand_intervals(raw["ind"].assign(NRI_CODE=lambda x: x["NRI_CODE"].astype(str).str.strip()),
                         ["NRI_CODE", "IND_NAME"], lo=START_MI, hi=END_MI)
ind_m = ind_m.sort_values("_from").drop_duplicates(["NRI_CODE", "mi"], keep="last").drop(columns="_from")

# 時価総額・リターン
mcap = raw["mcap"][["security_code", "month", "market_cap"]].copy()
mcap["security_code"] = norm_code(mcap["security_code"])
mcap["mi"] = period_to_mi(mcap["month"])
mcap = mcap.drop_duplicates(["security_code", "mi"])[["security_code", "mi", "market_cap"]]
rets = raw["ret"][["security_code", "month", "ret"]].copy()
rets["security_code"] = norm_code(rets["security_code"])
rets["mi"] = period_to_mi(rets["month"])
rets = rets.drop_duplicates(["security_code", "mi"])[["security_code", "mi", "ret"]]

univ = (topix_m.merge(code_m, on=["NRI_CODE", "mi"], how="left")
        .merge(ind_m, on=["NRI_CODE", "mi"], how="left")
        .merge(mcap, on=["security_code", "mi"], how="left"))
nxt = rets.rename(columns={"ret": "ret_fwd"}).assign(mi=lambda x: x["mi"] - 1)      # 翌月リターンを組成月に合わせる
univ = univ.merge(nxt, on=["security_code", "mi"], how="left")

print("【送付③】結合の診断（TOPIX構成 企業×月 に対する割合）")
print(f"  証券コードあり : {univ['security_code'].notna().mean():.1%}")
print(f"  時価総額あり   : {univ['market_cap'].notna().mean():.1%}   ← 低い場合は証券コードの形式（4桁/5桁）を確認")
print(f"  業種あり       : {univ['IND_NAME'].notna().mean():.1%}")
print(f"  翌月リターンあり: {univ['ret_fwd'].notna().mean():.1%}")

is_fin = univ["IND_NAME"].fillna("").apply(lambda s: any(k in s for k in EXCLUDE_INDUSTRY_KEYWORDS))
univ = univ[~is_fin & (univ["market_cap"] > 0)].copy()
print(f"金融除外・時価総額あり: {len(univ):,} 行, 月平均 {univ.groupby('mi').size().mean():.0f} 社")

## 4. 財務指標の作成と時点整合

**本決算（SETTLEMENT_FLG=1, 12か月決算）** の連結財務から指標を作り、**決算が公表された後**にだけ使います。

- 公表日 = 決算発表日（ANNOUNCE）→ 無ければ決算短信入手日（RCVD）→ 両方無ければ決算期末＋2か月（月末）
- 組成月 t では「公表日が月 t の月末以前」の最新の本決算を使う（`MAX_STALE_MONTHS` を超えて古いものは使わない）

| 指標 | 定義 |
|---|---|
| ROE | 当期純利益 ÷ 株主資本（期首・期末平均。期首が無ければ期末） |
| ROA / 営業利益/総資産 / 売上総利益/総資産 | 各利益 ÷ 総資産（期末） |
| 資産成長率 | 総資産 ÷ 前期の総資産 − 1 |
| アクルーアル | (Δ売上債権 + Δ棚卸資産 − Δ買入債務) ÷ 平均総資産（運転資本ベース） |
| 現金/総資産 | 現金・預金 ÷ 総資産 |
| B/P | 株主資本 ÷ 組成月の時価総額（単位は不明でも、順位で使うので影響しない） |

株主資本は 株主資本合計 → 純資産合計 → 資本合計 の順に補完。会計基準が複数あれば `CTYPE_PRIORITY` の順に1つを採用。

In [ ]:
det = raw["details"].copy()
det["NRI_CODE"] = det["NRI_CODE"].astype(str).str.strip()
for c in ["FTERM", "CTYPE", "SETTLEMENT_FLG", "NMONTH"]:
    det[c] = num(det[c])
det["ann_date"] = ymd_to_date(det["ANNOUNCE"]).fillna(ymd_to_date(det["RCVD"]))

# --- 本決算の財務 ---
key = ["NRI_CODE", "FTERM", "CTYPE"]
d1 = det[(det["SETTLEMENT_FLG"] == 1) & (det["NMONTH"] == 12)][key + ["ann_date"]].drop_duplicates(key)
pl = raw["pl"].copy()
bs = raw["bs"].copy()
for t in (pl, bs):
    t["NRI_CODE"] = t["NRI_CODE"].astype(str).str.strip()
    for c in t.columns:
        if c != "NRI_CODE":
            t[c] = num(t[c])
fund = d1.merge(pl.drop_duplicates(key), on=key, how="inner").merge(bs.drop_duplicates(key), on=key, how="left")
fund["_prio"] = fund["CTYPE"].map({c: i for i, c in enumerate(CTYPE_PRIORITY)}).fillna(99)
fund = fund.sort_values("_prio").drop_duplicates(["NRI_CODE", "FTERM"]).drop(columns="_prio")

# 公表日（無ければ決算期末＋2か月の月末）
fterm_mi = ym_to_mi(fund["FTERM"])
fallback = mi_to_ts(fterm_mi + 2) + pd.offsets.MonthEnd(0)
fund["avail_date"] = fund["ann_date"].fillna(pd.Series(fallback.to_numpy(), index=fund.index))
fund["avail_mi"] = (fund["avail_date"].dt.year - 1970) * 12 + fund["avail_date"].dt.month - 1
fund["fterm_mi"] = fterm_mi
print(f"本決算の公表日：発表日/短信入手日あり {fund['ann_date'].notna().mean():.1%}、残りは決算期末+2か月で補完")

# 指標
fund["equity"] = fund["CAPSTTL"].fillna(fund["NTASTTL"]).fillna(fund["EQTY"])
fund = fund.sort_values(["NRI_CODE", "fterm_mi"])
g = fund.groupby("NRI_CODE")
prev_ok = (fund["fterm_mi"] - g["fterm_mi"].shift()) == 12
for c in ["equity", "TOTALBS", "ACRC", "INVENT", "ACPY"]:
    fund[c + "_prev"] = g[c].shift().where(prev_ok)
eq_avg = ((fund["equity"] + fund["equity_prev"]) / 2).where(fund["equity_prev"] > 0, fund["equity"])
ta_avg = ((fund["TOTALBS"] + fund["TOTALBS_prev"]) / 2).where(fund["TOTALBS_prev"] > 0, fund["TOTALBS"])
fund["roe"] = (fund["NETINCM"] / eq_avg).where(eq_avg > 0)
fund["roa"] = fund["NETINCM"] / fund["TOTALBS"].where(fund["TOTALBS"] > 0)
fund["opa"] = fund["OPINCM"] / fund["TOTALBS"].where(fund["TOTALBS"] > 0)
fund["gpa"] = fund["GPLS"] / fund["TOTALBS"].where(fund["TOTALBS"] > 0)
fund["asset_growth"] = fund["TOTALBS"] / fund["TOTALBS_prev"].where(fund["TOTALBS_prev"] > 0) - 1
fund["accrual"] = ((fund["ACRC"] - fund["ACRC_prev"]) + (fund["INVENT"] - fund["INVENT_prev"])
                   - (fund["ACPY"] - fund["ACPY_prev"])) / ta_avg
fund["cash_ta"] = fund["CASH"] / fund["TOTALBS"].where(fund["TOTALBS"] > 0)

FUND_VARS = ["roe", "roa", "opa", "gpa", "asset_growth", "accrual", "cash_ta", "equity"]
LABELS = {"roe": "ROE", "roa": "ROA", "opa": "営業利益/総資産", "gpa": "売上総利益/総資産",
          "asset_growth": "資産成長率", "accrual": "アクルーアル", "cash_ta": "現金/総資産", "bp": "B/P", "size": "時価総額"}
fund = fund.sort_values("avail_mi").drop_duplicates(["NRI_CODE", "avail_mi"], keep="last")
fund[["roe", "asset_growth", "accrual"]].describe(percentiles=[0.01, 0.5, 0.99]).T

In [ ]:
# 組成月 t 時点で公表済みの最新の本決算を付ける（merge_asof：公表月 ≤ t）
left = univ.assign(mi=univ["mi"].astype("int64")).sort_values("mi")
right = fund.dropna(subset=["avail_mi"])[["NRI_CODE", "avail_mi", "fterm_mi"] + FUND_VARS]
right = right.assign(avail_mi=right["avail_mi"].astype("int64")).sort_values("avail_mi")
panel = pd.merge_asof(left, right, left_on="mi", right_on="avail_mi", by="NRI_CODE", direction="backward")
panel = panel[(panel["mi"] - panel["avail_mi"]) <= MAX_STALE_MONTHS].copy()
panel["bp"] = (panel["equity"] / panel["market_cap"]).where(panel["equity"] > 0)
panel["size"] = panel["market_cap"]

# 翌月（t+1）に決算発表があるか：全決算（四半期含む）と本決算
ann = det.dropna(subset=["ann_date"]).copy()
ann["ann_mi"] = ((ann["ann_date"].dt.year - 1970) * 12 + ann["ann_date"].dt.month - 1).astype("int64")
ann_any = ann.groupby(["NRI_CODE", "ann_mi"]).size().rename("n").reset_index()
ann_ann = ann[ann["SETTLEMENT_FLG"] == 1].groupby(["NRI_CODE", "ann_mi"]).size().rename("n").reset_index()
for name, a in (("ann_any", ann_any), ("ann_annual", ann_ann)):
    a = a.assign(mi=a["ann_mi"] - 1)[["NRI_CODE", "mi"]].assign(**{name: True})
    panel = panel.merge(a, on=["NRI_CODE", "mi"], how="left")
    panel[name] = panel[name].fillna(False).astype(bool)

# 国内（月内）での順位・階層
panel = panel.dropna(subset=["roe"]).copy()
gm = panel.groupby("mi")
for v in ["roe", "asset_growth", "accrual", "gpa", "size", "bp", "cash_ta"]:
    panel[v + "_pct"] = gm[v].rank(pct=True)
TIERS = ["低収益", "中収益", "高収益"]
panel["tier"] = pd.cut(panel["roe_pct"], [0, TIER_CUTS[0], TIER_CUTS[1], 1.0], labels=TIERS,
                       include_lowest=True).astype("object")

# 調整済みリターン（3通り）
ok = panel["ret_fwd"].notna()
panel["size_b"] = np.ceil(panel["size_pct"] * 3).clip(1, 3)
panel["bp_b"] = np.ceil(panel["bp_pct"] * 3).clip(1, 3)
panel["ret_mkt"] = panel["ret_fwd"] - panel[ok].groupby("mi")["ret_fwd"].transform("mean")
panel["ret_size"] = panel["ret_fwd"] - panel[ok].groupby(["mi", "size_b"])["ret_fwd"].transform("mean")
panel["ret_sbp"] = panel["ret_fwd"] - panel[ok].groupby(["mi", "size_b", "bp_b"])["ret_fwd"].transform("mean")
RET_COLS = {"mkt": "ret_mkt", "size": "ret_size", "size_bp": "ret_sbp"}
ADJ_LABELS = {"mkt": "市場調整", "size": "サイズ調整", "size_bp": "サイズ×B/P調整"}
ADJ_LABEL = ADJ_LABELS[RET_ADJ]
panel["ret_adj"] = panel[RET_COLS[RET_ADJ]]          # 主な分析で使うリターン
print(f"主な分析のリターン: {ADJ_LABEL}（補足として 9-2章で3通りを比較）")
panel["date"] = mi_to_ts(panel["mi"]).to_numpy()
panel["year"] = panel["date"].dt.year

print(f"分析パネル: {len(panel):,} 行（企業×組成月）, {panel['date'].min():%Y-%m}〜{panel['date'].max():%Y-%m}, "
      f"月平均 {panel.groupby('mi').size().mean():.0f} 社")
print(f"翌月が決算発表月である割合: 全決算 {panel['ann_any'].mean():.1%} / 本決算 {panel['ann_annual'].mean():.1%}")

In [ ]:
# 決算発表の季節性（どの月に発表が集中しているか）
cal = panel.assign(cal_m=(panel["mi"] + 1) % 12 + 1)
season = cal.groupby("cal_m")[["ann_any", "ann_annual"]].mean()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(season.index - 0.2, season["ann_any"], 0.4, label="全決算（四半期含む）", color="#7f8c8d")
ax.bar(season.index + 0.2, season["ann_annual"], 0.4, label="本決算", color="#c0392b")
ax.set_xticks(range(1, 13))
ax.set_xlabel("リターンを測る月（t+1）")
ax.set_title("その月に決算発表がある企業の割合")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend()
plt.show()

## 5. 前提の確認：高ROE企業のうち、どんな企業の高収益が続かないか

毎年6月時点の階層で、高ROE企業を **資産成長率** と **アクルーアル** の高低（高ROE企業内の中央値で分割）に分け、
h年後も高ROEに留まる割合を比べます。仮説が正しければ、資産成長やアクルーアルが高い高ROE企業ほど続きません。

In [ ]:
snap = panel[panel["date"].dt.month == 6][["NRI_CODE", "year", "tier", "asset_growth_pct", "accrual_pct", "roe_pct"]].copy()
hi = snap[snap["tier"] == "高収益"].copy()
for v in ["asset_growth_pct", "accrual_pct"]:
    med = hi.groupby("year")[v].transform("median")
    hi[v.replace("_pct", "_grp")] = np.where(hi[v] > med, "高", "低")
    hi.loc[hi[v].isna(), v.replace("_pct", "_grp")] = np.nan

rows = {}
for h in [1, 3, 5]:
    fut = snap[["NRI_CODE", "year", "tier"]].rename(columns={"tier": "tier_fwd"}).assign(year=lambda x: x["year"] - h)
    m = hi.merge(fut, on=["NRI_CODE", "year"], how="left")
    m["stay"] = (m["tier_fwd"] == "高収益").astype(float)       # 除外・欠損は「残らなかった」扱い
    m = m[m["year"] + h <= snap["year"].max()]
    rows[(f"{h}年後", "高ROE全体")] = m["stay"].mean()
    for v, lab_ in (("asset_growth_grp", "資産成長"), ("accrual_grp", "アクルーアル")):
        for gname in ["低", "高"]:
            rows[(f"{h}年後", f"{lab_}が{gname}い")] = m.loc[m[v] == gname, "stay"].mean()
persist = pd.Series(rows).unstack(0)
display(persist.style.format("{:.1%}"))

fig, ax = plt.subplots(figsize=(9, 3.8))
for i, (idx, r) in enumerate(persist.iterrows()):
    ax.plot([1, 3, 5], r.values, marker="o", label=idx,
            ls="-" if "高い" in idx else ("--" if "低い" in idx else ":"), color=plt.get_cmap("tab10")(i))
ax.set_xticks([1, 3, 5])
ax.set_xlabel("年後")
ax.set_title("高ROE企業が高ROEに留まる割合（指数除外・データ欠損は留まらなかった扱い）")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend(fontsize=8)
plt.show()

## 6. 決算発表月とそれ以外の月のリターン（収益性階層別）

各月・各グループの平均リターン（`RET_ADJ` の方法で調整。既定は B/P を使わない市場調整）を計算し、全期間で平均します。
仮説が正しければ、**高収益企業のリターンは決算発表月に低く**（期待が裏切られる）、それ以外の月との差が大きくなります。

In [ ]:
def group_ts(d, group_cols, val="ret_adj", weight=None):
    x = d.dropna(subset=[val] + group_cols)
    w = x["size"] if weight == "VW" else pd.Series(1.0, index=x.index)
    x = x.assign(_w=w, _wv=w * x[val], _n=1)
    gg = x.groupby(["mi"] + group_cols)
    r = (gg["_wv"].sum() / gg["_w"].sum())[gg["_n"].sum() >= MIN_STOCKS]
    return r


rows = {}
for ann_col, ann_lab in (("ann_any", "全決算"), ("ann_annual", "本決算")):
    r = group_ts(panel.assign(ann=panel[ann_col].map({True: "発表月", False: "非発表月"})), ["tier", "ann"])
    for (t, a), s in r.groupby(level=["tier", "ann"]):
        rows[(ann_lab, t, a)] = ts_stats(s)
tbl6 = pd.DataFrame(rows).T
display(tbl6)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, ann_lab in zip(axes, ["全決算", "本決算"]):
    t = tbl6.loc[ann_lab]["平均(%/月)"].unstack()
    x = np.arange(len(TIERS))
    ax.bar(x - 0.2, t.reindex(TIERS)["発表月"], 0.4, color="#c0392b", label="決算発表月")
    ax.bar(x + 0.2, t.reindex(TIERS)["非発表月"], 0.4, color="#95a5a6", label="それ以外の月")
    ax.set_xticks(x)
    ax.set_xticklabels(TIERS)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"{ann_lab}：収益性階層別の月次リターン（{ADJ_LABEL}, %）")
    ax.legend()
plt.tight_layout()
plt.show()

## 7. 収益性ロングショートは決算発表月に集中しているか

毎月、**決算発表がある企業だけ**で「高収益 − 低収益」の平均リターン差を計算し、**発表がない企業だけ**での差と比べます。
（日本は3月決算が多く、発表企業がいる月といない月がはっきり分かれるため、同じ月どうしではなく、それぞれの月の平均を比べます）
差（発表月 − 非発表月）がマイナスなら、収益性ファクターのリターンが決算発表で削られていることになります。
（EW: 等ウェイト、VW: 時価総額加重。生のリターンを使用。リターン欠損銘柄は除外＝組成時ウェイトで再配分と同じ）

In [ ]:
def nw_se(x, lags=NW_LAGS):
    x = pd.Series(x).dropna()
    t = nw_tstat(x, lags)
    return abs(x.mean() / t) if t and np.isfinite(t) and t != 0 else np.nan


def diff_tstat(a, b):
    """2つの月次系列の平均の差の t 値（それぞれ Newey-West 標準誤差、独立と仮定）"""
    return (pd.Series(a).mean() - pd.Series(b).mean()) / np.sqrt(nw_se(a) ** 2 + nw_se(b) ** 2)


def ls_split(d, ann_col, weight, val="ret_fwd"):
    x = d.assign(ann=d[ann_col])
    r = group_ts(x, ["ann", "tier"], val=val, weight=weight)
    r = r.unstack("tier")
    spread = (r["高収益"] - r["低収益"]).unstack("ann")
    return spread.rename(columns={True: "発表月", False: "非発表月"})


rows = {}
spreads = {}
for ann_col, ann_lab in (("ann_any", "全決算"), ("ann_annual", "本決算")):
    for w in ["EW", "VW"]:
        sp = ls_split(panel, ann_col, w)
        spreads[(ann_lab, w)] = sp
        a_, n_ = sp["発表月"].dropna(), sp["非発表月"].dropna()
        rows[(ann_lab, w)] = {
            "高−低（発表企業）%/月": a_.mean() * 100, "t(発表)": nw_tstat(a_),
            "高−低（非発表企業）%/月": n_.mean() * 100, "t(非発表)": nw_tstat(n_),
            "差（発表−非発表）": (a_.mean() - n_.mean()) * 100, "t(差)": diff_tstat(a_, n_),
            "月数(発表/非発表)": f"{len(a_)}/{len(n_)}"}
tbl7 = pd.DataFrame(rows).T
display(tbl7)

# 暦月別（日本は3月決算が多いので5月・11月などに集中）
sp = spreads[("全決算", "EW")]
by_m = sp.groupby((sp.index + 1) % 12 + 1).mean() * 100
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(by_m.index - 0.2, by_m["発表月"], 0.4, color="#c0392b", label="決算発表企業での 高−低")
ax.bar(by_m.index + 0.2, by_m["非発表月"], 0.4, color="#95a5a6", label="非発表企業での 高−低")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(1, 13))
ax.set_xlabel("リターンを測る月")
ax.set_title("収益性ロングショート（高−低, EW）の暦月別平均（%/月）")
ax.legend()
plt.show()

## 8. 月次クロスセクション回帰：収益性 × 決算発表月

Engelberg, McLean & Pontiff (2018) と同じ形を月次で行います。全企業×月をプールし、**月の固定効果**（各月の平均を引く）を入れ、
標準誤差は月でクラスターします（同じ月の企業どうしの相関に対応）。

$$r_{i,t+1}^{\text{市場調整}} = a + b\,\text{ROE順位}_i + c\,\text{発表}_i + d\,(\text{ROE順位}_i \times \text{発表}_i) + \text{サイズとその発表交差項}$$

- 主な式は日米で作り方をそろえるため **B/P を入れない**。B/P とその交差項を加えた式は補足（バリューの影響を除いても残るか）

- **d**：決算発表月に、収益性のリターンへの効果がどれだけ変わるか。**d < 0** なら決算で高収益企業が売られる（期待外れ）
- 2本目の式は高収益企業だけで、資産成長・アクルーアルが高い企業が発表月に売られるか（持続しない高収益への失望）を見ます

In [ ]:
def demeaned_ols_cluster(d, y, xcols, cluster="mi"):
    """月で中心化（月固定効果）した OLS と、月クラスター頑健標準誤差"""
    x = d.dropna(subset=[y] + xcols).copy()
    for c in [y] + xcols:
        x[c] = x[c].astype(float) - x.groupby(cluster)[c].transform("mean")
    X = x[xcols].to_numpy()
    Y = x[y].to_numpy()
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ Y
    u = Y - X @ b
    S = np.zeros((len(xcols), len(xcols)))
    for _, idx in x.groupby(cluster).indices.items():
        s_ = X[idx].T @ u[idx]
        S += np.outer(s_, s_)
    se = np.sqrt(np.diag(XtX_inv @ S @ XtX_inv))
    with np.errstate(divide="ignore", invalid="ignore"):
        t = np.where(se > 1e-12, b / se, np.nan)
    return pd.DataFrame({"係数(%/月)": b * 100, "t値(月クラスター)": t}, index=xcols)


res8 = {}
for ann_col, ann_lab in (("ann_any", "全決算"), ("ann_annual", "本決算")):
    x = panel.copy()
    x["ann"] = x[ann_col].astype(float)
    x["roe_x_ann"] = x["roe_pct"] * x["ann"]
    x["size_x_ann"] = x["size_pct"] * x["ann"]
    x["bp_x_ann"] = x["bp_pct"] * x["ann"]
    res8[(ann_lab, "全企業")] = demeaned_ols_cluster(
        x, "ret_mkt", ["roe_pct", "ann", "roe_x_ann", "size_pct", "size_x_ann"])
    res8[(ann_lab, "全企業（B/P統制, 補足）")] = demeaned_ols_cluster(
        x, "ret_mkt", ["roe_pct", "ann", "roe_x_ann", "size_pct", "size_x_ann", "bp_pct", "bp_x_ann"])
    h = x[x["tier"] == "高収益"].copy()
    for v in ["asset_growth", "accrual"]:
        h[v + "_hi"] = (h[v + "_pct"] > h.groupby("mi")[v + "_pct"].transform("median")).astype(float)
        h[v + "_hi_x_ann"] = h[v + "_hi"] * h["ann"]
    res8[(ann_lab, "高収益企業のみ")] = demeaned_ols_cluster(
        h, "ret_mkt", ["ann", "asset_growth_hi", "asset_growth_hi_x_ann", "accrual_hi", "accrual_hi_x_ann", "size_pct"])
NAMES8 = {"roe_pct": "ROE順位", "ann": "発表月", "roe_x_ann": "ROE順位×発表月（d）", "size_pct": "サイズ順位",
          "bp_pct": "B/P順位", "size_x_ann": "サイズ×発表月", "bp_x_ann": "B/P×発表月",
          "asset_growth_hi": "資産成長が高い", "asset_growth_hi_x_ann": "資産成長が高い×発表月",
          "accrual_hi": "アクルーアルが高い", "accrual_hi_x_ann": "アクルーアルが高い×発表月"}
tbl8 = pd.concat({k: v.rename(index=NAMES8) for k, v in res8.items()})
tbl8

## 9. 高ROE企業を「続きそう / 続かなそう」に分けた比較

5章で高収益が続かなかった特徴（資産成長が高い・アクルーアルが高い）で高ROE企業を分け、
**決算発表月とそれ以外の月**のリターンを比べます。

仮説が正しければ：**続かなそうな高ROE企業**は決算発表月に大きく下がり、**続きそうな高ROE企業**は下がらない（または上がる）。

In [ ]:
h = panel[panel["tier"] == "高収益"].copy()
for v in ["asset_growth", "accrual"]:
    med = h.groupby("mi")[v + "_pct"].transform("median")
    h[v + "_grp"] = np.where(h[v + "_pct"] > med, "高", "低")
    h.loc[h[v + "_pct"].isna(), v + "_grp"] = np.nan
h["persist_grp"] = np.select(
    [(h["asset_growth_grp"] == "高") & (h["accrual_grp"] == "高"), (h["asset_growth_grp"] == "低") & (h["accrual_grp"] == "低")],
    ["続かなそう（資産成長・アクルーアルとも高）", "続きそう（両方とも低）"], default="その他")

rows = {}
for gcol, glab in (("asset_growth_grp", "資産成長"), ("accrual_grp", "アクルーアル"), ("persist_grp", "組み合わせ")):
    for ann_col, ann_lab in (("ann_any", "全決算"), ("ann_annual", "本決算")):
        r = group_ts(h.assign(ann=h[ann_col].map({True: "発表月", False: "非発表月"})), [gcol, "ann"])
        for (gv, a), s in r.groupby(level=[gcol, "ann"]):
            rows[(glab, ann_lab, gv, a)] = ts_stats(s)
tbl9 = pd.DataFrame(rows).T
display(tbl9)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2), sharey=True)
for ax, glab in zip(axes, ["資産成長", "アクルーアル", "組み合わせ"]):
    t = tbl9.loc[(glab, "全決算")]["平均(%/月)"].unstack()
    t = t.drop(index="その他", errors="ignore")
    x = np.arange(len(t))
    ax.bar(x - 0.2, t["発表月"], 0.4, color="#c0392b", label="決算発表月")
    ax.bar(x + 0.2, t["非発表月"], 0.4, color="#95a5a6", label="それ以外の月")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{glab}{i}" if glab != "組み合わせ" else i for i in t.index], fontsize=8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"高ROE企業：{glab}別（全決算, {ADJ_LABEL}, %/月）")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 9-2. 補足：リターンの調整方法による違い

6章・9章の主な結果を、3通りの調整（市場 / サイズ / サイズ×B/P）で並べます。

- **市場調整で決算発表月の下落が見えるが、サイズ×B/P調整で消える** → 決算で裏切られているのは「割高な高ROE企業」で、
  割高さ（低いB/P）そのものが問題の中心
- **どの調整でも残る** → 割高さとは別に、「持続しない高収益」そのものが期待を裏切っている

In [ ]:
rows = {}
for adj, col in RET_COLS.items():
    r = group_ts(panel.assign(ann=panel["ann_any"].map({True: "発表月", False: "非発表月"})), ["tier", "ann"], val=col)
    for (t, a), s_ in r.groupby(level=["tier", "ann"]):
        if t in ("高収益", "低収益"):
            rows[(ADJ_LABELS[adj], f"{t}・{a}")] = s_.mean() * 100
    r = group_ts(h.assign(ann=h["ann_any"].map({True: "発表月", False: "非発表月"})), ["persist_grp", "ann"], val=col)
    for (g_, a), s_ in r.groupby(level=["persist_grp", "ann"]):
        if g_ != "その他":
            rows[(ADJ_LABELS[adj], f"高ROE {g_.split('（')[0]}・{a}")] = s_.mean() * 100
adj_cmp = pd.Series(rows).unstack(0)[[ADJ_LABELS[k] for k in RET_COLS]]
print("【送付⑤】リターンの調整方法による比較：各グループの平均リターン（%/月, 全決算）")
display(adj_cmp.style.format("{:+.2f}"))

fig, ax = plt.subplots(figsize=(12, 4.2))
x = np.arange(len(adj_cmp))
for i, c in enumerate(adj_cmp.columns):
    ax.bar(x + (i - 1) * 0.27, adj_cmp[c], 0.27, label=c, color=["#2c3e50", "#7f8c8d", "#e67e22"][i])
ax.set_xticks(x)
ax.set_xticklabels(adj_cmp.index, rotation=25, ha="right", fontsize=8)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("リターンの調整方法による違い（%/月）")
ax.legend()
plt.tight_layout()
plt.show()

## 10. 決算で明らかになった ROE の変化とリターン

本決算の発表月について、**新しく公表された ROE − それまでの ROE（ΔROE）** と、その月のリターン（`RET_ADJ` の方法で調整）を見ます。

- グループ（収益性階層 × 資産成長の高低）ごとに、平均 ΔROE と平均リターンを比べる
- 「高ROE・高成長」で ΔROE が最も下がり、リターンも最も低ければ、**持続しない高収益が決算で明らかになり株価が下がる** という流れを示せる
- 回帰で、グループの違いが ΔROE で説明されるか（ΔROE を入れるとグループ効果が消えるか）も確認

In [ ]:
new = fund[["NRI_CODE", "avail_mi", "roe"]].rename(columns={"roe": "roe_new"}).astype({"avail_mi": "int64"})
ev = panel[panel["ann_annual"]].copy()
ev["avail_next"] = (ev["mi"] + 1).astype("int64")
ev = ev.merge(new, left_on=["NRI_CODE", "avail_next"], right_on=["NRI_CODE", "avail_mi"], how="left", suffixes=("", "_n"))
ev = ev.dropna(subset=["roe_new", "ret_adj"]).copy()
ev["d_roe"] = ev["roe_new"] - ev["roe"]
ev["ag_grp"] = np.where(ev["asset_growth_pct"] > ev.groupby(["mi", "tier"])["asset_growth_pct"].transform("median"), "高成長", "低成長")
print(f"本決算イベント: {len(ev):,} 件")

grp = ev.groupby(["tier", "ag_grp"]).agg(平均ΔROE=("d_roe", "median"), 平均リターン=("ret_adj", "mean"), 件数=("ret_adj", "size"))
display(grp.style.format({"平均ΔROE": "{:+.4f}", "平均リターン": "{:+.2%}", "件数": "{:,.0f}"}))

fig, ax = plt.subplots(figsize=(8, 5))
colors = {"低収益": "#95a5a6", "中収益": "#e9c46a", "高収益": "#c0392b"}
for (t, a), r in grp.iterrows():
    ax.scatter(r["平均ΔROE"], r["平均リターン"], s=120, color=colors[t], marker="o" if a == "低成長" else "^")
    ax.annotate(f"{t}・{a}", (r["平均ΔROE"], r["平均リターン"]), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("決算で明らかになった ROE の変化（中央値）")
ax.set_ylabel(f"決算発表月のリターン（平均, {ADJ_LABEL}）")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("ROE の変化と決算発表月のリターン（本決算）")
plt.show()

In [ ]:
ev["hi_hiag"] = ((ev["tier"] == "高収益") & (ev["ag_grp"] == "高成長")).astype(float)
ev["hi_loag"] = ((ev["tier"] == "高収益") & (ev["ag_grp"] == "低成長")).astype(float)
ev["lo"] = (ev["tier"] == "低収益").astype(float)
ev["d_roe_pct"] = ev.groupby("mi")["d_roe"].rank(pct=True)
m1 = demeaned_ols_cluster(ev, "ret_adj", ["hi_hiag", "hi_loag", "lo"])
# 注：月固定効果のため、係数は同じ月の中ROE企業との差（%）
m2 = demeaned_ols_cluster(ev, "ret_adj", ["hi_hiag", "hi_loag", "lo", "d_roe_pct"])
NAMES10 = {"hi_hiag": "高ROE・高成長", "hi_loag": "高ROE・低成長", "lo": "低ROE", "d_roe_pct": "ΔROE（順位）"}
tbl10 = pd.concat({"(1) グループのみ": m1, "(2) ＋ΔROE": m2}, axis=1).rename(index=NAMES10)
print("基準＝中ROE企業。(2) で高ROE・高成長の係数がゼロに近づけば、その低リターンは『決算で明らかになったROEの低下』で説明される")
tbl10

## 11. 期間別

ガバナンス改革（2014〜2015年）や東証のPBR改善要請（2023年）の前後で、決算発表月への集中が変わったかを確認します。

In [ ]:
rows = {}
for pname, (y0, y1) in PERIODS.items():
    sub = panel[(panel["year"] >= y0) & (panel["year"] <= y1)]
    if sub.empty:
        continue
    for w in ["EW", "VW"]:
        sp = ls_split(sub, "ann_any", w)
        a_, n_ = sp["発表月"].dropna(), sp["非発表月"].dropna()
        rows[(pname, w)] = {"高−低（発表）%/月": a_.mean() * 100, "高−低（非発表）%/月": n_.mean() * 100,
                            "差": (a_.mean() - n_.mean()) * 100, "t(差)": diff_tstat(a_, n_), "月数": len(a_) + len(n_)}
    hh = h[(h["year"] >= y0) & (h["year"] <= y1)]
    r = group_ts(hh.assign(ann=hh["ann_any"].map({True: "発表月", False: "非発表月"})), ["asset_growth_grp", "ann"])
    for (gv, a), s in r.groupby(level=["asset_growth_grp", "ann"]):
        rows[(pname, f"高ROE・資産成長{gv}・{a}")] = {"高−低（発表）%/月": np.nan, "高−低（非発表）%/月": np.nan,
                                                   "差": s.mean() * 100, "t(差)": nw_tstat(s), "月数": len(s)}
tbl11 = pd.DataFrame(rows).T
print(f"上段：収益性ロングショート（全決算）。下段：高ROE企業の資産成長別 平均リターン（%/月, {ADJ_LABEL}）を『差』列に表示")
tbl11

## 12. まとめ（自動抽出）

In [ ]:
def pick(tbl, idx, col):
    try:
        return tbl.loc[idx, col]
    except KeyError:
        return np.nan


lines = ["【送付④】まとめ（TOPIX・決算発表月）", f"リターンの調整方法: {ADJ_LABEL}", ""]
lines.append("■ 前提：高ROE企業が高ROEに留まる割合（1年 / 3年 / 5年後）")
for idx, r in persist.iterrows():
    lines.append(f"  - {idx}: " + " / ".join(f"{v:.0%}" for v in r.values))
lines.append("\n■ 収益性ロングショート（高−低）の決算発表月への集中")
for (a, w), r in tbl7.iterrows():
    lines.append(f"  - {a}・{w}: 発表企業 {r['高−低（発表企業）%/月']:+.2f}%/月 / 非発表企業 {r['高−低（非発表企業）%/月']:+.2f}%/月"
                 f" → 差 {r['差（発表−非発表）']:+.2f}%（t={r['t(差)']:.1f}）")
lines.append("\n■ 回帰：ROE順位×発表月（d）")
for k in [("全決算", "全企業"), ("本決算", "全企業")]:
    lines.append(f"  - {k[0]}: d = {pick(tbl8, (k[0], k[1], 'ROE順位×発表月（d）'), '係数(%/月)'):+.2f}%"
                 f"（t={pick(tbl8, (k[0], k[1], 'ROE順位×発表月（d）'), 't値(月クラスター)'):.1f}）")
lines.append(f"\n■ 高ROE企業の決算発表月リターン（全決算, {ADJ_LABEL}）")
for gv in ["続かなそう（資産成長・アクルーアルとも高）", "続きそう（両方とも低）"]:
    a = pick(tbl9, ("組み合わせ", "全決算", gv, "発表月"), "平均(%/月)")
    b = pick(tbl9, ("組み合わせ", "全決算", gv, "非発表月"), "平均(%/月)")
    lines.append(f"  - {gv}: 発表月 {a:+.2f}% / それ以外 {b:+.2f}%")
lines.append("\n■ 決算で明らかになった ROE 変化（本決算）")
for (t, a), r in grp.iterrows():
    if t == "高収益":
        lines.append(f"  - 高ROE・{a}: ΔROE中央値 {r['平均ΔROE']:+.3f} / 発表月リターン {r['平均リターン']:+.2%}")
lines.append(f"  - 高ROE・高成長の係数（中ROE比, %）: ΔROEなし {pick(tbl10, '高ROE・高成長', ('(1) グループのみ', '係数(%/月)')):+.2f}"
             f" → ΔROEあり {pick(tbl10, '高ROE・高成長', ('(2) ＋ΔROE', '係数(%/月)')):+.2f}")
print("\n".join(lines))

### 読み方

| 結果 | 解釈 |
|---|---|
| 5章で、資産成長・アクルーアルが高い高ROE企業ほど高ROEに留まらない | 「続かない高収益」を事前に見分けられる（前提が成り立つ） |
| 7章の「差（発表−非発表）」がマイナス、8章の d がマイナス | 収益性ファクターのリターンが決算発表で削られている＝**期待外れ（ミスプライシング）**。リスクでは説明しにくい |
| 9章で「続かなそう」な高ROE企業だけが発表月に大きくマイナス | 削られているのは**持続しない高収益**に対する過大な期待 |
| 10章で高ROE・高成長の ΔROE が最も低く、ΔROE を入れると係数が消える | 低リターンは「決算で ROE の低下が明らかになったこと」で説明される |
| 11章で改革後に集中が弱まる | 市場の期待形成（または企業の資本配分）が変わった可能性 |

**限界**
- 月次リターンなので、決算発表以外のニュースも混ざる（日次リターンがあれば発表日前後3日で精度が上がる）
- 決算発表日が欠損している場合は短信入手日、さらに無ければ決算期末＋2か月で補完している
- 「続きそう／続かなそう」の分け方は資産成長・アクルーアルの中央値。他の指標（営業CF、売上総利益など）で頑健性を確認するとよい